In [26]:
import yfinance as yf
import pandas as pd

In [27]:
#Download 3 years of daily historical data for NVIDIA
df=yf.download("NVDA", start="2023-01-01", end="2026-09-14",interval="1d",multi_level_index=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed


In [31]:
import  talib #Assuming standard wrapper setup

close_prices = df["Close"].values.flatten()  # Flatten the array to 1D if necessary
#calculate a 50-day SMA and a 14-period RSI
df["SMA_50"]=talib.SMA(close_prices, timeperiod=50)
df["RSI_14"]=talib.RSI(close_prices, timeperiod=14)
#Drop missing values caused by indicator lag windows
df.dropna(inplace=True)

In [30]:
import numpy as np #for mathematical signal generation and returns calculations
#Use .iloc[:,0] to force the 2D "close" dataframe into a 1D series

#Generate continuous binary trading positions(1=Holdlong,0=cash)
df["Signal"]=np.where((close_series > df["SMA_50"]) & (df["RSI_14"] < 65),1,0)
#Calculate daily asset log returns
df["Market_Returns"]=np.log(close_series / close_series.shift(1))
#Calculate strategy returns by multiplying market returns by the previous day's signal
df["Strategy_Returns"]=df["Market_Returns"] * df["Signal"].shift(1) #Shift signal to avoid lookahead bias

In [10]:
import vectorbt as vbt #for backtesting and evaluation
#Run the simulation assuming a $10,000 initial capital and no transaction costs
portfolio=vbt.Portfolio.from_signals(df["Close"].iloc[:,0], entries=(df["Signal"]==1), exits=(df["Signal"]==0),init_cash=10000,freq="d")
#print metrics like sharpe ratio, max drawdown, and total return
print(portfolio.stats())

Start                               2023-03-15 00:00:00
End                                 2026-09-11 00:00:00
Period                                877 days 00:00:00
Start Value                                     10000.0
End Value                                  16444.995308
Total Return [%]                              64.449953
Benchmark Return [%]                         803.841072
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                              31.607003
Max Drawdown Duration                 238 days 00:00:00
Total Trades                                         67
Total Closed Trades                                  66
Total Open Trades                                     1
Open Trade PnL                               895.057897
Win Rate [%]                                  48.484848
Best Trade [%]                                23.761748
Worst Trade [%]                              -15

In [13]:
print(df[["Close", "SMA_50"]].head(60))  # Look at the first 60 rows
print("\nMissing values count:")
print(df[["Close", "SMA_50"]].isna().sum())
print("\nTotal dataset rows:", len(df))


Price           Close     SMA_50
Ticker           NVDA           
Date                            
2023-03-15  24.151369  20.389570
2023-03-16  25.460217  20.613428
2023-03-17  25.643629  20.832304
2023-03-20  25.818077  21.064316
2023-03-21  26.116131  21.290449
2023-03-22  26.384285  21.506616
2023-03-23  27.104992  21.731596
2023-03-24  26.694296  21.946529
2023-03-27  26.447083  22.146351
2023-03-28  26.326468  22.336026
2023-03-29  26.898645  22.521139
2023-03-30  27.296389  22.720685
2023-03-31  27.689144  22.940285
2023-04-03  27.876549  23.142225
2023-04-04  27.366167  23.306968
2023-04-05  26.795977  23.458871
2023-04-06  26.951485  23.612729
2023-04-10  27.491768  23.767844
2023-04-11  27.083063  23.903562
2023-04-12  26.411198  24.049823
2023-04-13  26.379297  24.187971
2023-04-14  26.673365  24.303974
2023-04-17  26.916592  24.409573
2023-04-18  27.579494  24.540569
2023-04-19  27.842657  24.677048
2023-04-20  27.018270  24.775431
2023-04-21  27.033222  24.873475
2023-04-24

In [24]:
import talib
# 1. Flatten df["Close"] to a 1D series so TA-Lib can read it
close_prices = df["Close"]


# Calculate the 50-day simple moving average
df["SMA_50"] = talib.SMA(close_prices, timeperiod=50)

# Drop rows where SMA hasn't started calculating yet
df_clean = df.dropna(subset=["SMA_50"])

In [33]:
import plotly.graph_objects as go #for interactive plotting
fig=go.Figure()
#Add the close price trace
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Close"], mode="lines", name="NVDAClose Price", line=dict(color="#76B900")))
#Add the 50-day SMA trace
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["SMA_50"], mode="lines", name="50-Day SMA", line=dict(color="#1f77b4", dash="dash")))
fig.update_layout(
    title="NVIDIA Corporation (NVDA) - Close Price vs 50-Day SMA",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()